In [ ]:
import numpy as np
import pandas as pd
import matplotlib as plt
import seaborn as sns

orders = pd.read_csv('../data/raw/orders.csv')
aisles = pd.read_csv('../data/raw/aisles.csv')
order_products_prior = pd.read_csv('../data/raw/order_products__prior.csv')
order_products_train = pd.read_csv('../data/raw/order_products__train.csv')
departments = pd.read_csv('../data/raw/departments.csv')
products = pd.read_csv('../data/raw/products.csv')


print(orders.shape)
print(aisles.shape)
print(order_products_prior.shape)
print(order_products_train.shape)
print(departments.shape)
print(products.shape)




(3421083, 7)
(134, 2)
(32434489, 4)
(1384617, 4)
(21, 2)
(49688, 4)


In [ ]:

print(orders['eval_set'].value_counts())
print(orders.head())
print('Order lists:', orders.columns.tolist())
print('Aisles lists:', aisles.columns.tolist())
print('Departments lists:', departments.columns.tolist())
print('Product lists:',  products.columns.tolist())
print('Order prior lists:', order_products_prior.columns.tolist())
print('Order train lists:', order_products_train.columns.to_list())

eval_set
prior    3214874
train     131209
test       75000
Name: count, dtype: int64
   order_id  user_id eval_set  order_number  order_dow  order_hour_of_day  \
0   2539329        1    prior             1          2                  8   
1   2398795        1    prior             2          3                  7   
2    473747        1    prior             3          3                 12   
3   2254736        1    prior             4          4                  7   
4    431534        1    prior             5          4                 15   

   days_since_prior_order  
0                     NaN  
1                    15.0  
2                    21.0  
3                    29.0  
4                    28.0  
Order lists: ['order_id', 'user_id', 'eval_set', 'order_number', 'order_dow', 'order_hour_of_day', 'days_since_prior_order']
Aisles lists: ['aisle_id', 'aisle']
Departments lists: ['department_id', 'department']
Product lists: ['product_id', 'product_name', 'aisle_id', 'department_i

In [ ]:
orders_prior = orders[orders['eval_set'] == 'prior']
orders_train = orders[orders['eval_set'] == 'train']

print("Prior orders:", orders_prior.shape)
print("Train orders:", orders_train.shape)


Prior orders: (3214874, 7)
Train orders: (131209, 7)


In [ ]:
basket_size = order_products_prior.groupby('order_id')['product_id'].count().reset_index()

basket_size.columns = ['order_id', 'basket_size']

order_prior_basket = orders_prior.merge(basket_size, on='order_id')

print(order_prior_basket.head())
user_features = order_prior_basket.groupby('user_id').agg(
    total_orders = ('order_id', 'count'),
    avg_days_between_order = ('days_since_prior_order', 'mean'),
    median_days_between_order = ('days_since_prior_order', 'median'),
    avg_basket_size = ('basket_size', 'mean'),
    favourite_hour = ('order_hour_of_day', lambda x : x.mode()[0]),
    favourite_dow = ('order_dow', lambda x : x.mode()[0]),
).reset_index()

print(user_features.shape)
print(user_features.head())

   order_id  user_id eval_set  order_number  order_dow  order_hour_of_day  \
0   2539329        1    prior             1          2                  8   
1   2398795        1    prior             2          3                  7   
2    473747        1    prior             3          3                 12   
3   2254736        1    prior             4          4                  7   
4    431534        1    prior             5          4                 15   

   days_since_prior_order  basket_size  
0                     NaN            5  
1                    15.0            6  
2                    21.0            5  
3                    29.0            5  
4                    28.0            8  
(206209, 7)
   user_id  total_orders  avg_days_between_order  median_days_between_order  \
0        1            10               19.555556                       20.0   
1        2            14               15.230769                       13.0   
2        3            12               12.

In [ ]:
# Step 1 — add user_id to order_products_prior
prior_with_user = order_products_prior.merge(
    orders_prior[['order_id', 'user_id']], 
    on='order_id'
)

# Step 2 — build product features
product_features = prior_with_user.groupby('product_id').agg(
    product_total_orders=('order_id', 'count'),
    product_reorder_rate=('reordered', 'mean'),
    product_unique_users=('user_id', 'nunique'),
    product_avg_cart_position=('add_to_cart_order', 'mean')
).reset_index()

print(product_features.shape)
print(product_features.head())

(49677, 5)
   product_id  product_total_orders  product_reorder_rate  \
0           1                  1852              0.613391   
1           2                    90              0.133333   
2           3                   277              0.732852   
3           4                   329              0.446809   
4           5                    15              0.600000   

   product_unique_users  product_avg_cart_position  
0                   716                   5.801836  
1                    78                   9.888889  
2                    74                   6.415162  
3                   182                   9.507599  
4                     6                   6.466667  


In [ ]:
prior_with_user = prior_with_user.merge(
    orders_prior[['order_id', 'order_number']],
    on='order_id'
)

user_product_features = prior_with_user.groupby(
    ['user_id', 'product_id']
).agg(
    up_times_bought=('order_id', 'count'),
    up_reorder_rate=('reordered', 'mean'),
    up_avg_cart_pos=('add_to_cart_order', 'mean'),
    up_last_order=('order_number', 'max')
).reset_index()

print(user_product_features.shape)
print(user_product_features.head())


(13307953, 6)
   user_id  product_id  up_times_bought  up_reorder_rate  up_avg_cart_pos  \
0        1         196               10         0.900000         1.400000   
1        1       10258                9         0.888889         3.333333   
2        1       10326                1         0.000000         5.000000   
3        1       12427               10         0.900000         3.300000   
4        1       13032                3         0.666667         6.333333   

   up_last_order  
0             10  
1             10  
2              5  
3             10  
4             10  


In [ ]:
# order_products = orders.merge(order_products_prior, on= 'order_id')

user1_orders = orders[orders['user_id'] == 1]['order_id']

user1_products = order_products_prior[order_products_prior['order_id'].isin(user1_orders)]

print(len(user1_products))


popular_day = orders.groupby('order_hour_of_day')['user_id'].count()
print(popular_day.idxmax())

board = order_products_prior.merge(
    products[['product_id', 'department_id']],  # only needed columns!
    on='product_id'
)
big_board = board.merge(departments, on= 'department_id')
print(big_board.groupby('department')['reordered'].mean().sort_values(ascending=False))

order_per_user = orders.groupby('user_id')['order_id'].count()
print((order_per_user > 20).sum())


59
10
department
dairy eggs         0.669969
beverages          0.653460
produce            0.649913
bakery             0.628141
deli               0.607719
pets               0.601285
babies             0.578971
bulk               0.577040
snacks             0.574180
alcohol            0.569924
meat seafood       0.567674
breakfast          0.560922
frozen             0.541885
dry goods pasta    0.461076
canned goods       0.457405
other              0.407980
household          0.402178
missing            0.395849
international      0.369229
pantry             0.346721
personal care      0.321129
Name: reordered, dtype: float64
50731


In [ ]:
print(order_products_train.head())
print(order_products_train.shape)
print(order_products_train['reordered'].value_counts())

   order_id  product_id  add_to_cart_order  reordered
0         1       49302                  1          1
1         1       11109                  2          1
2         1       10246                  3          0
3         1       49683                  4          0
4         1       43633                  5          1
(1384617, 4)
reordered
1    828824
0    555793
Name: count, dtype: int64


In [ ]:
train_with_user = order_products_train.merge(
    orders_train[['order_id', 'user_id']],
    on= 'order_id'
)
print(train_with_user.shape)
print(train_with_user.head())

(1384617, 5)
   order_id  product_id  add_to_cart_order  reordered  user_id
0         1       49302                  1          1   112108
1         1       11109                  2          1   112108
2         1       10246                  3          0   112108
3         1       49683                  4          0   112108
4         1       43633                  5          1   112108


In [ ]:
# Step 1 - left merge
df_final = train_with_user.merge(
    user_product_features, on=['user_id', 'product_id'],
    how='left'
).merge(
    user_features, on='user_id', how='left'
).merge(
    product_features, on='product_id', how='left'
)

print("Shape:", df_final.shape)
print("\nMissing values:")
print(df_final.isnull().sum())

Shape: (1384617, 19)

Missing values:
order_id                          0
product_id                        0
add_to_cart_order                 0
reordered                         0
user_id                           0
up_times_bought              555793
up_reorder_rate              555793
up_avg_cart_pos              555793
up_last_order                555793
total_orders                      0
avg_days_between_order            0
median_days_between_order         0
avg_basket_size                   0
favourite_hour                    0
favourite_dow                     0
product_total_orders              9
product_reorder_rate              9
product_unique_users              9
product_avg_cart_position         9
dtype: int64


In [ ]:
# User-product features → fill with 0 (no history = zero)
df_final['up_times_bought'] = df_final['up_times_bought'].fillna(0)
df_final['up_reorder_rate'] = df_final['up_reorder_rate'].fillna(0)
df_final['up_last_order'] = df_final['up_last_order'].fillna(0)

# Cart position → fill with median (reasonable estimate)
df_final['up_avg_cart_pos'] = df_final['up_avg_cart_pos'].fillna(
    df_final['up_avg_cart_pos'].median()
)

# Product features → fill with median (9 unknown products)
df_final['product_total_orders'] = df_final['product_total_orders'].fillna(
    df_final['product_total_orders'].median()
)
df_final['product_reorder_rate'] = df_final['product_reorder_rate'].fillna(
    df_final['product_reorder_rate'].median()
)
df_final['product_unique_users'] = df_final['product_unique_users'].fillna(
    df_final['product_unique_users'].median()
)
df_final['product_avg_cart_position'] = df_final['product_avg_cart_position'].fillna(
    df_final['product_avg_cart_position'].median()
)

# Verify
print("Remaining nulls:", df_final.isnull().sum().sum())
print("Shape:", df_final.shape)

Remaining nulls: 0
Shape: (1384617, 19)


In [ ]:


# Step 1 — get all user-product pairs from prior
all_user_products = prior_with_user[['user_id', 'product_id']].drop_duplicates()
print("All user-product pairs:", all_user_products.shape)

# Step 2 — get what they actually bought in train order
train_pairs = train_with_user[['user_id', 'product_id']].copy()
train_pairs['reordered'] = 1

# Step 3 — left merge to label all pairs
all_user_products = all_user_products.merge(
    train_pairs,
    on=['user_id', 'product_id'],
    how='left'
)
all_user_products['reordered'] = all_user_products['reordered'].fillna(0).astype(int)

print("Shape:", all_user_products.shape)
print("Target distribution:")
print(all_user_products['reordered'].value_counts())

All user-product pairs: (13307953, 2)
Shape: (13307953, 3)
Target distribution:
reordered
0    12479129
1      828824
Name: count, dtype: int64


In [ ]:
# Merge all features
df_final_correct = all_user_products.merge(
    user_product_features, on=['user_id', 'product_id'], how='left'
).merge(
    user_features, on='user_id', how='left'
).merge(
    product_features, on='product_id', how='left'
)

# Fill NaN
df_final_correct['up_times_bought'] = df_final_correct['up_times_bought'].fillna(0)
df_final_correct['up_reorder_rate'] = df_final_correct['up_reorder_rate'].fillna(0)
df_final_correct['up_last_order'] = df_final_correct['up_last_order'].fillna(0)
df_final_correct['up_avg_cart_pos'] = df_final_correct['up_avg_cart_pos'].fillna(
    df_final_correct['up_avg_cart_pos'].median()
)

print("Shape:", df_final_correct.shape)
print("Nulls:", df_final_correct.isnull().sum().sum())

MemoryError: Unable to allocate 812. MiB for an array with shape (8, 13307953) and data type int64

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import time

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

# Scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

Train: (1107693, 14)
Test: (276924, 14)


In [ ]:
# Model 1 — Logistic Regression
print("Training Logistic Regression...")
start = time.time()
lr = LogisticRegression(random_state=42)
lr.fit(X_train_scaled, y_train)
lr_time = time.time() - start

lr_pred = lr.predict(X_test_scaled)
lr_acc = accuracy_score(y_test, lr_pred)
print(f"LR Accuracy: {lr_acc:.3f} (trained in {lr_time:.1f}s)")

# Model 2 — Random Forest
print("\nTraining Random Forest...")
start = time.time()
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1  # use all CPU cores!
)
rf.fit(X_train, y_train)  # RF doesn't need scaling
rf_time = time.time() - start

rf_pred = rf.predict(X_test)
rf_acc = accuracy_score(y_test, rf_pred)
print(f"RF Accuracy: {rf_acc:.3f} (trained in {rf_time:.1f}s)")

Training Logistic Regression...
LR Accuracy: 1.000 (trained in 1.4s)

Training Random Forest...
RF Accuracy: 1.000 (trained in 37.2s)


In [ ]:
# Check feature correlations with target
correlations = X.corrwith(y).abs().sort_values(ascending=False)
print(correlations)

up_reorder_rate              0.677582
up_last_order                0.520665
up_times_bought              0.433668
product_reorder_rate         0.303943
total_orders                 0.223947
product_avg_cart_position    0.188371
avg_days_between_order       0.172439
median_days_between_order    0.166614
product_unique_users         0.156961
product_total_orders         0.152628
up_avg_cart_pos              0.148210
avg_basket_size              0.120444
favourite_hour               0.021106
favourite_dow                0.007168
dtype: float64


In [ ]:
# Check if model is just predicting all 1s or all 0s
print("LR predictions distribution:")
print(pd.Series(lr_pred).value_counts())

print("\nRF predictions distribution:")
print(pd.Series(rf_pred).value_counts())

# Check a sample of predictions vs actual
comparison = pd.DataFrame({
    'actual': y_test.values[:20],
    'lr_pred': lr_pred[:20],
    'rf_pred': rf_pred[:20]
})
print(comparison)

# Verify manually
correct_lr = (lr_pred == y_test.values).sum()
total = len(y_test)
print(f"LR: {correct_lr}/{total} = {correct_lr/total:.6f}")

LR predictions distribution:
1    165658
0    111266
Name: count, dtype: int64

RF predictions distribution:
1    165658
0    111266
Name: count, dtype: int64
    actual  lr_pred  rf_pred
0        0        0        0
1        1        1        1
2        0        0        0
3        0        0        0
4        0        0        0
5        1        1        1
6        1        1        1
7        1        1        1
8        1        1        1
9        0        0        0
10       1        1        1
11       0        0        0
12       1        1        1
13       0        0        0
14       1        1        1
15       0        0        0
16       0        0        0
17       1        1        1
18       1        1        1
19       0        0        0
LR: 276924/276924 = 1.000000


In [ ]:
# Check this immediately
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

# Check if test set is subset of training set
print("\nX_train first row:", X_train.iloc[0].values)
print("X_test first row:", X_test.iloc[0].values)

X_train shape: (1107693, 14)
X_test shape: (276924, 14)
y_train shape: (1107693,)
y_test shape: (276924,)

X_train first row: [  9.           0.88888889   1.55555556  15.          15.
  10.21428571   7.           7.6         14.           4.
 469.           0.62899787 174.           7.60341151]
X_test first row: [0.00000000e+00 0.00000000e+00 6.84210526e+00 0.00000000e+00
 4.00000000e+00 2.66666667e+01 3.00000000e+01 9.00000000e+00
 7.00000000e+00 5.00000000e+00 1.52400000e+03 3.93044619e-01
 9.25000000e+02 8.87204724e+00]


In [ ]:
# Check df_model index
print("df_model index unique:", df_model.index.nunique())
print("df_model shape:", df_model.shape)
# These two numbers should be equal!

df_model index unique: 1384617
df_model shape: (1384617, 17)


In [ ]:
# Try a completely fresh simple test
# Use only 1000 rows to debug

sample = df_model.sample(1000, random_state=42)
X_sample = sample.drop(columns=['reordered', 'user_id', 'product_id'])
y_sample = sample['reordered']

X_s_train, X_s_test, y_s_train, y_s_test = train_test_split(
    X_sample, y_sample, test_size=0.2, random_state=42
)

scaler_s = StandardScaler()
X_s_train_scaled = scaler_s.fit_transform(X_s_train)
X_s_test_scaled = scaler_s.transform(X_s_test)

lr_s = LogisticRegression(random_state=42)
lr_s.fit(X_s_train_scaled, y_s_train)
lr_s_pred = lr_s.predict(X_s_test_scaled)

print(f"Sample accuracy: {accuracy_score(y_s_test, lr_s_pred):.3f}")
print(pd.Series(lr_s_pred).value_counts())

Sample accuracy: 0.955
1    105
0     95
Name: count, dtype: int64


In [ ]:
# Use stratified split — ensures balanced classes
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y  # ← ensures same class ratio in train/test
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lr = LogisticRegression(random_state=42)
lr.fit(X_train_scaled, y_train)
pred = lr.predict(X_test_scaled)

print(f"Accuracy: {accuracy_score(y_test, pred):.3f}")
print(classification_report(y_test, pred))

Accuracy: 1.000
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    111159
           1       1.00      1.00      1.00    165765

    accuracy                           1.00    276924
   macro avg       1.00      1.00      1.00    276924
weighted avg       1.00      1.00      1.00    276924



In [ ]:
# Remove the suspicious feature
X_no_leak = X.drop(columns=['up_reorder_rate','up_last_order', 'product_reorder_rate', 'product_avg_cart_position', 'favourite_hour', 'favourite_dow', 'avg_basket_size','product_unique_users', 'product_total_orders','total_orders','avg_days_between_order'])

X_train, X_test, y_train, y_test = train_test_split(
    X_no_leak, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lr = LogisticRegression(random_state=42)
lr.fit(X_train_scaled, y_train)
pred = lr.predict(X_test_scaled)

print(f"Accuracy without up_reorder_rate: {accuracy_score(y_test, pred):.3f}")

Accuracy without up_reorder_rate: 1.000


In [ ]:
# Check how many unique feature combinations exist
print("Unique feature rows:", X.drop_duplicates().shape[0])
print("Total rows:", X.shape[0])

# Check remaining columns
print("\nRemaining columns:", X_no_leak.columns.tolist())

# Try with just ONE feature
X_single = X[['up_times_bought']]
X_tr, X_te, y_tr, y_te = train_test_split(
    X_single, y, test_size=0.2, random_state=42, stratify=y
)
lr_single = LogisticRegression()
lr_single.fit(X_tr, y_tr)
print(f"\nAccuracy with ONLY up_times_bought: {accuracy_score(y_te, lr_single.predict(X_te)):.3f}")

# Try with just zeros
X_zero = pd.DataFrame({'dummy': [0]*len(y)})
X_tr, X_te, y_tr, y_te = train_test_split(
    X_zero, y, test_size=0.2, random_state=42, stratify=y
)
lr_zero = LogisticRegression()
lr_zero.fit(X_tr, y_tr)
print(f"Accuracy predicting ALL 1s: {y_te.mean():.3f}")

Unique feature rows: 1384552
Total rows: 1384617

Remaining columns: ['up_times_bought', 'up_avg_cart_pos', 'median_days_between_order']

Accuracy with ONLY up_times_bought: 1.000
Accuracy predicting ALL 1s: 0.599
